# Task 3: Citation Span Extraction - BERT Inference

**Model:** bert-base-uncased fine-tuned on citation span extraction

**Input:** `.label` files from test set

**Output:** Predicted span for each citation in each document

**Notes:**
- Dùng softmax độc lập cho start/end (không dùng pipeline vì score underflow)
- Chỉ tìm span trong phần context (sequence_id=1), bỏ qua phần question
- Đảm bảo start <= end

---

## 1. Setup

In [1]:
import os, json
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print(f"CUDA: {torch.cuda.is_available()}")

CUDA: True


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if dirs:
        print("dirs:", dirs)
    if files:
        print("files:", files[:20])
    print("-" * 50)

/kaggle/input
dirs: ['datasets', 'notebooks']
--------------------------------------------------
/kaggle/input/datasets
dirs: ['tathiyennhi']
--------------------------------------------------
/kaggle/input/datasets/tathiyennhi
dirs: ['task3-citation-span-extraction']
--------------------------------------------------
/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction
dirs: ['task3']
--------------------------------------------------
/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3
dirs: ['test_silver', 'val', 'test_gold', 'train']
--------------------------------------------------
/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/test_silver
files: ['27135.label', '24831.in', '50677.in', '19806.label', '28867.in', '16176.label', '40364.in', '23928.label', '45182.in', '32080.label', '20529.in', '23397.in', '57198.in', '45323.label', '25324.label', '36419.label', '13956.label', '60341.label', '13528.in', '24220.label']
--------

## 2. Config

In [3]:
import os

# Đường dẫn Test (Lấy từ log của bạn - mình chọn test_silver làm ví dụ)
TEST_DIR = "/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/test_silver"

# Đường dẫn Output (Luôn để ở working để có quyền ghi)
OUTPUT_DIR = "/kaggle/working/predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Đường dẫn Model (Dựa trên ảnh Notebooks trước đó và quy luật log này)
# Bạn hãy chạy dòng dưới đây để xác nhận MODEL_DIR nếu đoạn này vẫn lỗi:
MODEL_DIR = "/kaggle/input/notebooks/tathiyennhi/task3-bert-citation-span-v2/task3_bert_final_v2"

# Cấu hình Model
MAX_LENGTH = 512
MAX_ANSWER_LEN = 100  # token

# Tạo thư mục Output
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Model Dir:  {MODEL_DIR}")
print(f"✅ Test Dir:   {TEST_DIR}")
print(f"✅ Output Dir: {OUTPUT_DIR} (Đã sẵn sàng để lưu file)")

if os.path.exists(MODEL_DIR):
    print("📂 Thư mục model tồn tại!")
    print("Các file bên trong:", os.listdir(MODEL_DIR))
else:
    print("❌ CẢNH BÁO: Không tìm thấy thư mục model. Hãy kiểm tra lại đường dẫn!")

✅ Model Dir:  /kaggle/input/notebooks/tathiyennhi/task3-bert-citation-span-v2/task3_bert_final_v2
✅ Test Dir:   /kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/test_silver
✅ Output Dir: /kaggle/working/predictions (Đã sẵn sàng để lưu file)
📂 Thư mục model tồn tại!
Các file bên trong: ['config.json', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']


## 3. Load Model

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

print(f"✅ Model loaded on {DEVICE}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model loaded on cuda


## 4. Inference Function

In [5]:
def predict_span(question, context):
    """
    Predict citation span using independent softmax for start/end.
    Returns (pred_text, score, char_start, char_end)
    """
    inputs = tokenizer(
        question, context,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )
    encoded = tokenizer(
        question, context,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH
    )
    sequence_ids = encoded.sequence_ids()
    offset_mapping = inputs.pop("offset_mapping")[0]
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_probs = F.softmax(outputs.start_logits[0], dim=0)
    end_probs   = F.softmax(outputs.end_logits[0], dim=0)

    best_score, best_start, best_end = -1, 0, 0
    for s in range(len(start_probs)):
        for e in range(s, min(s + MAX_ANSWER_LEN, len(end_probs))):
            if sequence_ids[s] != 1 or sequence_ids[e] != 1:
                continue
            score = start_probs[s].item() * end_probs[e].item()
            if score > best_score:
                best_score, best_start, best_end = score, s, e

    char_start = offset_mapping[best_start][0].item()
    char_end   = offset_mapping[best_end][1].item()
    pred_text  = context[char_start:char_end]

    return pred_text, best_score, char_start, char_end


print("✅ Inference function defined")

✅ Inference function defined


## 5. Run Inference on Test Set

In [6]:
label_files = sorted(Path(TEST_DIR).glob("*.label"))
print(f"📊 Test files: {len(label_files):,}")

all_predictions = []
skipped = 0

for i, label_file in enumerate(label_files):
    if (i + 1) % 100 == 0:
        print(f"⏳ {i+1:,}/{len(label_files):,}")

    try:
        with open(label_file) as f:
            data = json.load(f)
    except:
        skipped += 1
        continue

    doc_id  = data.get("doc_id", label_file.stem)
    context = data.get("text", "")
    if not context:
        skipped += 1
        continue

    doc_preds = {"doc_id": doc_id, "predictions": []}

    for span_info in data.get("citation_spans", []):
        citation_id = span_info.get("citation_id", "")
        question    = f"What does citation {citation_id} support?"

        pred_text, score, char_start, char_end = predict_span(question, context)

        doc_preds["predictions"].append({
            "citation_id": citation_id,
            "pred_span_text": pred_text,
            "pred_s_span": char_start,
            "pred_e_span": char_end,
            "score": round(score, 6)
        })

    all_predictions.append(doc_preds)

# Save
output_path = f"{OUTPUT_DIR}/predictions.json"
with open(output_path, "w") as f:
    json.dump(all_predictions, f, indent=2)

print(f"\n✅ Done: {len(all_predictions):,} docs | Skipped: {skipped}")
print(f"✅ Saved to: {output_path}")

📊 Test files: 2,417
⏳ 100/2,417
⏳ 200/2,417
⏳ 300/2,417
⏳ 400/2,417
⏳ 500/2,417
⏳ 600/2,417
⏳ 700/2,417
⏳ 800/2,417
⏳ 900/2,417
⏳ 1,000/2,417
⏳ 1,100/2,417
⏳ 1,200/2,417
⏳ 1,300/2,417
⏳ 1,400/2,417
⏳ 1,500/2,417
⏳ 1,600/2,417
⏳ 1,700/2,417
⏳ 1,800/2,417
⏳ 1,900/2,417
⏳ 2,000/2,417
⏳ 2,100/2,417
⏳ 2,200/2,417
⏳ 2,300/2,417
⏳ 2,400/2,417

✅ Done: 2,417 docs | Skipped: 0
✅ Saved to: /kaggle/working/predictions/predictions.json


## 6. Sample Output

In [7]:
for doc in all_predictions[:2]:
    print(f"\ndoc_id: {doc['doc_id']}")
    for p in doc['predictions']:
        print(f"  {p['citation_id']} | score={p['score']:.4f}")
        print(f"  → {p['pred_span_text'][:100]}...")


doc_id: 10014
  [CITATION_1] | score=0.7439
  → We restricted these samples to the 282,836 and 304,294 live births that occurred at least 1 y and at...

doc_id: 10049
  [CITATION_1] | score=0.9842
  → Quantifications were done in duplicate and mean values and standard deviation were calculated for ea...


In [8]:
import json
import os

# 1. Tự động tìm file trainer_state.json trong toàn bộ thư mục Input
state_file_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'trainer_state.json' in files:
        state_file_path = os.path.join(root, 'trainer_state.json')
        break

# 2. Khởi tạo các giá trị mặc định
train_loss = "N/A"
epoch = "N/A"
f1 = 0.0
em = 0.0
eval_loss = "N/A"

# 3. Nếu tìm thấy file, bốc dữ liệu ra ngay
if state_file_path:
    with open(state_file_path, 'r') as f:
        data = json.load(f)
    
    # Lấy thông số training cuối cùng
    log_history = data.get('log_history', [])
    for entry in reversed(log_history):
        # Lấy Train Loss
        if 'loss' in entry and train_loss == "N/A":
            train_loss = f"{entry['loss']:.6f}"
        
        # Lấy Validation Metrics (F1, EM) - Đây là số bạn cần nhất!
        if 'eval_f1' in entry and f1 == 0.0:
            f1 = entry['eval_f1']
            em = entry.get('eval_exact_match', 0)
            eval_loss = f"{entry.get('eval_loss', 0):.6f}"
            epoch = f"{entry.get('epoch', 0):.2f}"
            
    print(f"✅ Đã tìm thấy và trích xuất dữ liệu từ: {state_file_path}")
else:
    print("❌ Không tìm thấy file trainer_state.json trong các Notebook đã add.")

# 4. In bảng kết quả cuối cùng
summary = f"""
============================================================
       EXPERIMENT RESULTS - Task 3 Citation Span Extraction
============================================================

[Training Summary]
  Epochs:          {epoch}
  Train Loss:      {train_loss}
  Eval Loss:       {eval_loss}

[Final Metrics - SQuAD-style F1]
  F1 Score:        {f1:.4f} ({f1*100:.2f}%)
  Exact Match:     {em:.4f} ({em*100:.2f}%)

[System Info]
  Model:           bert-base-uncased
  Project:         task3-citation-span-extraction
============================================================
"""

print(summary)

# 5. Lưu lại để sau này chỉ cần mở file .txt xem
os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/final_report.txt", "w") as f:
    f.write(summary)

✅ Đã tìm thấy và trích xuất dữ liệu từ: /kaggle/input/notebooks/tathiyennhi/task3-bert-citation-span-v2/resume_ckpt/trainer_state.json

       EXPERIMENT RESULTS - Task 3 Citation Span Extraction

[Training Summary]
  Epochs:          2.53
  Train Loss:      0.150826
  Eval Loss:       0.195591

[Final Metrics - SQuAD-style F1]
  F1 Score:        0.9781 (97.81%)
  Exact Match:     0.9192 (91.92%)

[System Info]
  Model:           bert-base-uncased
  Project:         task3-citation-span-extraction



## 7. Experiment Results

In [9]:
import json
import os

# ============================================================
# Export experiment summary from trainer_state.json
# ============================================================

# 1. Tự động tìm file trainer_state.json trong /kaggle/input
state_file_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "trainer_state.json" in files:
        state_file_path = os.path.join(root, "trainer_state.json")
        break

# 2. Khởi tạo giá trị mặc định
train_loss = None
epoch = None
eval_loss = None
f1 = None
em = None

# 3. Đọc dữ liệu từ trainer_state.json nếu tìm thấy
if state_file_path:
    with open(state_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    log_history = data.get("log_history", [])

    # Duyệt ngược để lấy kết quả cuối cùng gần nhất
    for entry in reversed(log_history):
        if train_loss is None and "loss" in entry and isinstance(entry["loss"], (int, float)):
            train_loss = entry["loss"]

        if f1 is None and "eval_f1" in entry:
            f1 = entry.get("eval_f1", 0.0)
            em = entry.get("eval_exact_match", 0.0)
            eval_loss = entry.get("eval_loss", None)
            epoch = entry.get("epoch", None)

        # Nếu đã đủ dữ liệu thì dừng
        if train_loss is not None and f1 is not None:
            break

    print(f"Đã tìm thấy trainer_state.json tại: {state_file_path}")
else:
    print("Không tìm thấy trainer_state.json trong /kaggle/input")

# 4. Format dữ liệu để hiển thị đẹp hơn
train_loss_str = f"{train_loss:.6f}" if train_loss is not None else "N/A"
epoch_str = f"{epoch:.2f}" if epoch is not None else "N/A"
eval_loss_str = f"{eval_loss:.6f}" if isinstance(eval_loss, (int, float)) else "N/A"
f1_str = f"{f1:.4f}" if isinstance(f1, (int, float)) else "N/A"
f1_pct_str = f"{f1 * 100:.2f}%" if isinstance(f1, (int, float)) else "N/A"
em_str = f"{em:.4f}" if isinstance(em, (int, float)) else "N/A"
em_pct_str = f"{em * 100:.2f}%" if isinstance(em, (int, float)) else "N/A"

# 5. Tạo nội dung báo cáo gọn và sạch hơn
summary = f"""
EXPERIMENT SUMMARY
Task 3: Citation Span Extraction

1. Model Configuration
- Model name      : bert-base-uncased
- Project name    : task3-citation-span-extraction

2. Training Information
- Epoch           : {epoch_str}
- Training loss   : {train_loss_str}
- Validation loss : {eval_loss_str}

3. Evaluation Metrics
- Metric type     : SQuAD-style F1 (word-level token overlap)
- F1 score        : {f1_str} ({f1_pct_str})
- Exact Match     : {em_str} ({em_pct_str})

4. Source
- Trainer state   : {state_file_path if state_file_path else "Not found"}
"""

print(summary)

# 6. Lưu file kết quả
output_dir = "/kaggle/working/results"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "final_report.txt")
with open(file_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(f"Đã lưu báo cáo tại: {file_path}")

Đã tìm thấy trainer_state.json tại: /kaggle/input/notebooks/tathiyennhi/task3-bert-citation-span-v2/resume_ckpt/trainer_state.json

EXPERIMENT SUMMARY
Task 3: Citation Span Extraction

1. Model Configuration
- Model name      : bert-base-uncased
- Project name    : task3-citation-span-extraction

2. Training Information
- Epoch           : 2.53
- Training loss   : 0.150826
- Validation loss : 0.195591

3. Evaluation Metrics
- Metric type     : SQuAD-style F1 (word-level token overlap)
- F1 score        : 0.9781 (97.81%)
- Exact Match     : 0.9192 (91.92%)

4. Source
- Trainer state   : /kaggle/input/notebooks/tathiyennhi/task3-bert-citation-span-v2/resume_ckpt/trainer_state.json

Đã lưu báo cáo tại: /kaggle/working/results/final_report.txt
